In [1]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)


In [2]:
df = pd.read_csv('filtered.csv')
df.head()

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_63,il_bsarg_64,il_bsarg_70,scenario,original_scenario,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,0.00,0.00,0.00,h0,h2,0,0,7,42,54
1,870600,87060001,0,0,0,870600,87060001,32,1,0,...,469.93,469.93,469.93,h0,h2,0,0,11,44,50
2,870700,87070001,0,0,0,870700,87070001,61,0,0,...,1747.61,1747.61,1747.61,h0,h0,1,0,7,27,52
3,870900,87090001,0,0,0,870900,87090001,33,0,0,...,100.00,100.00,100.00,h0,h2,0,0,10,37,51
4,871700,87170002,0,0,87170001,871700,87170002,61,0,0,...,312.50,312.50,312.50,h0,h2,0,0,7,40,51


In [3]:
def map_lhw(row):
    # Ensure the scenario prefix 'h' is included when constructing the column name
    scenario_prefix = 'h' if not row["scenario"].startswith('h') else ''
    scenario_column_name = f'lhw_{scenario_prefix}{row["scenario"]}'
    return row[scenario_column_name]

# Apply the corrected function
df['lhw_scenario'] = df.apply(map_lhw, axis=1)

# Print a sample to verify the column has been created correctly
print(df[['idperson', 'scenario', 'lhw_scenario']].head())


   idperson scenario  lhw_scenario
0  87010001       h0             0
1  87060001       h0             0
2  87070001       h0             0
3  87090001       h0             0
4  87170002       h0             0


In [4]:
def utility(c, l, beta1, beta2, gamma1, gamma2):
    epsilon = 1e-6  # Small constant to ensure c is strictly positive
    c_adjusted = np.maximum(c + epsilon, epsilon)  # Ensure c is positive
    l_adjusted = np.maximum(l, epsilon)  # Ensure l is positive, though l seems not to have this issue
    c_transformed = (np.power(c_adjusted, gamma1) - 1) / gamma1 if gamma1 != 0 else np.log(c_adjusted)
    l_transformed = (np.power(l_adjusted, gamma2) - 1) / gamma2 if gamma2 != 0 else np.log(l_adjusted)
    return beta1 * c_transformed + beta2 * l_transformed


In [5]:
# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2= params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll




In [6]:
# Jacobian 
#box_cox
def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p, df), epsilon)
    return grad

initial_params = [0.365, 0.8031, 0.25, -0.002757]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df,), method='Newton-CG', jac=jacobian)

print(f'Optimization result: {result}')

Optimization result:  message: Warning: CG iterations didn't converge. The Hessian is not positive definite.
 success: False
  status: 3
     fun: 10769.055921559893
       x: [ 3.657e-01 -7.480e-03  2.506e-01  1.412e+00]
     nit: 33
     jac: [ 0.000e+00  1.483e+04  0.000e+00 -1.007e+03]
    nfev: 62
    njev: 317
    nhev: 0


In [19]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.342e-05]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df,), method='Newton-CG', jac=jacobian)

print(f'Optimization result box cox cleaned data : {result}')


Optimization result box cox cleaned data :  message: Warning: Desired error not necessarily achieved due to precision loss.
 success: False
  status: 2
     fun: 7938.0045979762945
       x: [ 7.660e-01  9.197e-01  1.135e-01 -1.993e-03  4.342e-05]
     nit: 0
     jac: [ 0.000e+00  0.000e+00 -1.123e+01 -1.261e+03  7.472e+03]
    nfev: 19
    njev: 9
    nhev: 0


In [15]:
# Identify individuals with any negative 'ils_udb_yds' values
negative_c_ids = df[df['ils_udb_yds'] < 0]['idperson'].unique()
# Identify individuals with any negative 'ils_udb_yds' values
negative_l_ids = df[df['lhw'] > 80 ]['idperson'].unique()


print(f"Number of individuals with negative leisure: {len(negative_l_ids)}")
print(f"Number of individuals with negative consumption: {len(negative_c_ids)}")


Number of individuals with negative leisure: 3
Number of individuals with negative consumption: 0


In [16]:
# Filter long_df to exclude all rows belonging to individuals identified in step 1
df_filtered = df[~df['idperson'].isin(negative_c_ids)]
# Filter long_df to exclude all rows belonging to individuals identified in step 1
df_filtered = df[~df['idperson'].isin(negative_l_ids)]


# Verify the removal
print(f"Original dataframe size: {df.shape}")
print(f"Filtered dataframe size: {df_filtered.shape}")


Original dataframe size: (18320, 350)
Filtered dataframe size: (18308, 350)


In [20]:
# boox_cox cleaned
def utility(c, l, beta1, beta2, gamma1, gamma2):
    epsilon = 1e-6  # Small constant to ensure c is strictly positive
    c_adjusted = np.maximum(c + epsilon, epsilon)  # Ensure c is positive
    l_adjusted = np.maximum(l, epsilon)  # Ensure l is positive, though l seems not to have this issue
    c_transformed = (np.power(c_adjusted, gamma1) - 1) / gamma1 if gamma1 != 0 else np.log(c_adjusted)
    l_transformed = (np.power(l_adjusted, gamma2) - 1) / gamma2 if gamma2 != 0 else np.log(l_adjusted)
    return beta1 * c_transformed + beta2 * l_transformed

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2= params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll



# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params = [0.365, 0.8031, 0.25, -0.002757]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result box cox cleaned data : {result}')

Optimization result box cox cleaned data :  message: Warning: CG iterations didn't converge. The Hessian is not positive definite.
 success: False
  status: 3
     fun: 11809.369186390035
       x: [ 3.591e-01 -1.530e+00  2.502e-01 -4.248e-02]
     nit: 4
     jac: [ 0.000e+00  1.481e+02  0.000e+00 -6.386e+02]
    nfev: 9
    njev: 186
    nhev: 0


In [22]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [7.660e-01,  9.197e-01,  1111.135e-01, 21.993e03,  4.342e-05]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Warning: Desired error not necessarily achieved due to precision loss.
 success: False
  status: 2
     fun: 595236211608.4166
       x: [ 8.498e-01  7.373e+00  8.972e+01  2.135e+04 -1.442e+03]
     nit: 3
     jac: [ 0.000e+00  0.000e+00  1.638e+05  2.515e+07 -4.105e+07]
    nfev: 16
    njev: 64
    nhev: 0


In [23]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l

In [24]:

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

In [25]:
def calc_hessian(params, df):
    # This function should compute the Hessian matrix of your total likelihood function
    # with respect to the parameters.
    # For now, it returns a placeholder identity matrix of appropriate size
    return np.eye(len(params))

In [28]:
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll
def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

In [32]:
# Newton-Raphson Update Function
def newton_raphson_update(params, df):
    grad = jacobian(params, df)  # Use your existing gradient calculation
    hessian = calc_hessian(params, df)  # Placeholder for your Hessian calculation
    params_update = np.linalg.solve(hessian, -grad)  # Solving Hx = -grad for x
    return params + params_update

# Initialization and Iteration
initial_params = np.array([ 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.342e-05])
max_iterations = 100
convergence_threshold = 1e-6

params = initial_params
for iteration in range(max_iterations):
    params_new = newton_raphson_update(params, df_filtered)  # Use your data frame
    if np.linalg.norm(params_new - params) < convergence_threshold:
        print(f"Converged in {iteration + 1} iterations")
        break
    params = params_new

print(f"Final Parameters: {params}")


Converged in 4 iterations
Final Parameters: [ 7.66000000e-01  9.19700000e-01  5.86816930e+02 -3.14271135e+07
  4.96406282e+07]


In [33]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 7.66000000e-01, -5.36870912e+08, -3.19376887e+05, -1.82378654e+08
  8.08172738e+07]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Warning: Desired error not necessarily achieved due to precision loss.
 success: False
  status: 2
     fun: 1.2344236687920844e+16
       x: [ 7.660e-01 -5.369e+08 -3.194e+05 -1.824e+08  8.082e+07]
     nit: 2
     jac: [ 0.000e+00 -5.000e-01  0.000e+00  0.000e+00 -2.684e+08]
    nfev: 25
    njev: 18
    nhev: 0


In [35]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 7.66000000e-01,  9.19700000e-01,  5.86816930e+02, -3.14271135e+07,
  4.96406282e+07]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 1.0218990777957534e+16
       x: [ 6.766e+00 -7.612e+00  5.868e+02 -3.143e+07  4.964e+07]
     nit: 4
     jac: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.342e+08]
    nfev: 5
    njev: 19
    nhev: 0


In [36]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.342e-05]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 7909.4970816856185
       x: [ 7.660e-01  9.197e-01  1.135e-01 -1.993e-03  4.356e-05]
     nit: 1
     jac: [ 0.000e+00  0.000e+00 -5.867e+02 -3.017e+04 -3.576e+05]
    nfev: 3
    njev: 4
    nhev: 0


In [38]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.342e-05]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 7909.4970816856185
       x: [ 7.660e-01  9.197e-01  1.135e-01 -1.993e-03  4.356e-05]
     nit: 1
     jac: [ 0.000e+00  0.000e+00 -5.867e+02 -3.017e+04 -3.576e+05]
    nfev: 3
    njev: 4
    nhev: 0


In [37]:
### quadratic  theory !!! uncleaned

def utility(c, l, alpha , beta , gamma, delta,zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  alpha*c + beta *l - 0.5 * (gamma * c**2 + delta * l**2 + 2*zeta*c*l )

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha , beta , gamma, delta, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, alpha , beta , gamma, delta, zeta),
        utility(yds, 80 - lhw_scenario, alpha , beta , gamma, delta, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 7.66000000e-01,  9.19700000e-01,  5.86816930e-02, 3.14271135e-01,
  4.96406282e-01]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Warning: Desired error not necessarily achieved due to precision loss.
 success: False
  status: 2
     fun: 291709.87845248583
       x: [ 7.660e-01  9.199e-01  5.868e-02  3.052e-01 -6.977e-03]
     nit: 3
     jac: [ 3.906e-02 -1.779e+04 -7.812e-03  1.004e+06  2.290e+05]
    nfev: 22
    njev: 178
    nhev: 0


In [39]:
### quadratic  theory !!! rapjhson params

def utility(c, l, alpha , beta , gamma, delta,zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  alpha*c + beta *l - 0.5 * (gamma * c**2 + delta * l**2 + 2*zeta*c*l )
# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha , beta , gamma, delta, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, alpha , beta , gamma, delta, zeta),
        utility(yds, 80 - lhw_scenario, alpha , beta , gamma, delta, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]


def calc_hessian(params, df):
    # This function should compute the Hessian matrix of your total likelihood function
    # with respect to the parameters.
    # For now, it returns a placeholder identity matrix of appropriate size
    return np.eye(len(params))

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Newton-Raphson Update Function
def newton_raphson_update(params, df):
    grad = jacobian(params, df)  # Use your existing gradient calculation
    hessian = calc_hessian(params, df)  # Placeholder for your Hessian calculation
    params_update = np.linalg.solve(hessian, -grad)  # Solving Hx = -grad for x
    return params + params_update

# Initialization and Iteration
initial_params = np.array([ 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.342e-05])
max_iterations = 100
convergence_threshold = 1e-6

params = initial_params
for iteration in range(max_iterations):
    params_new = newton_raphson_update(params, df_filtered)  # Use your data frame
    if np.linalg.norm(params_new - params) < convergence_threshold:
        print(f"Converged in {iteration + 1} iterations")
        break
    params = params_new

print(f"Final Parameters: {params}")


Final Parameters: [ 5.94125000e-01 -2.97485830e+05 -5.36870872e+08 -4.47117970e+08
  2.63168457e+08]


In [40]:
### quadratic  theory !!! uncleaned

def utility(c, l, alpha , beta , gamma, delta,zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  alpha*c + beta *l - 0.5 * (gamma * c**2 + delta * l**2 + 2*zeta*c*l )

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha , beta , gamma, delta, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, alpha , beta , gamma, delta, zeta),
        utility(yds, 80 - lhw_scenario, alpha , beta , gamma, delta, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 5.94125000e-01, -2.97485830e+05, -5.36870872e+08, -4.47117970e+08,
  2.63168457e+08]
# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Warning: CG iterations didn't converge. The Hessian is not positive definite.
 success: False
  status: 3
     fun: 5.7359511248319064e+16
       x: [ 5.941e-01 -2.975e+05 -5.369e+08 -4.471e+08  2.632e+08]
     nit: 1
     jac: [ 0.000e+00  0.000e+00 -4.000e+00  3.732e+06  2.243e+08]
    nfev: 7
    njev: 207
    nhev: 0


In [44]:
## best specification 

alpha , beta , gamma, delta,zeta = 5.94125000e-01, -2.97485830e+05, -5.36870872e+08, -4.47117970e+08, 2.63168457e+08

def du_dc(c,l ): 
 return alpha - gamma*c + 2*zeta*l

def du_dl(c,l ): 
 return beta - delta*l + 2*zeta*c


# Apply the derivatives to each row in the DataFrame
df_filtered['du/dc'] = df_filtered.apply(lambda row: du_dc(row['ils_udb_yds'], 80 - row['lhw']), axis=1)
df_filtered['du/dL'] = df_filtered.apply(lambda row: du_dl(row['ils_udb_yds'], 80 - row['lhw']), axis=1)

# Find IDs where du/dc or du/dL are negative
negative_du_dc = df[df['du/dc'] < 0]['idperson'].tolist()
negative_du_dl = df[df['du/dL'] < 0]['idperson'].tolist()

print("IDs with negative du/dc:", negative_du_dc)
print("IDs with negative du/dL:", negative_du_dl)

C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\1916234459.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['du/dc'] = df_filtered.apply(lambda row: du_dc(row['ils_udb_yds'], 80 - row['lhw']), axis=1)


IDs with negative du/dc: []
IDs with negative du/dL: []


C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\1916234459.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['du/dL'] = df_filtered.apply(lambda row: du_dl(row['ils_udb_yds'], 80 - row['lhw']), axis=1)


In [49]:
df_filtered['du/dL']

0        4.514287e+11
1        4.584295e+11
2        8.245376e+11
3        4.505957e+11
4        4.339011e+11
             ...     
18315    1.190807e+12
18316    1.197484e+12
18317    1.209474e+12
18318    1.311717e+12
18319    9.279499e+11
Name: du/dL, Length: 18308, dtype: float64

In [58]:
params_quad = [ 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.356e-05]
alpha , beta , gamma, delta, zeta = 7.660e-01,  9.197e-01,  1.135e-01, -1.993e-03,  4.356e-05

def du1_dc(c,l ): 
 return alpha + 2* beta *c + zeta*l

def du1_dl(c,l ): 
 return gamma  + 2 * delta  *l  + zeta*c 


# Apply the derivatives to each row in the DataFrame
df_filtered['du1/dc'] = df_filtered.apply(lambda row: du_dc(row['ils_udb_yds'], 80 - row['lhw']), axis=1)
df_filtered['du1/dL'] = df_filtered.apply(lambda row: du_dl(row['ils_udb_yds'], 80 - row['lhw']), axis=1)




C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\954833132.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['du1/dc'] = df_filtered.apply(lambda row: du_dc(row['ils_udb_yds'], 80 - row['lhw']), axis=1)
C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\954833132.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['du1/dL'] = df_filtered.apply(lambda row: du_dl(row['ils_udb_yds'], 80 - row['lhw']), axis=1)


In [66]:
df_filtered[df_filtered['du/dL'] < 0]

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw_scenario,du/dc,du/dL,du1/dc,du1/dL


In [69]:
### quadratic  theory !!! rapjhson params

def utility(c, l, alpha , beta , gamma, delta,zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  alpha*c + beta *l - 0.5 * (gamma * c**2 + delta * l**2 + 2*zeta*c*l )
# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha , beta , gamma, delta, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, alpha , beta , gamma, delta, zeta),
        utility(yds, 80 - lhw_scenario, alpha , beta , gamma, delta, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]


def calc_hessian(params, df):
    # This function should compute the Hessian matrix of your total likelihood function
    # with respect to the parameters.
    # For now, it returns a placeholder identity matrix of appropriate size
    return np.eye(len(params))

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Newton-Raphson Update Function
def newton_raphson_update(params, df):
    grad = jacobian(params, df)  # Use your existing gradient calculation
    hessian = calc_hessian(params, df)  # Placeholder for your Hessian calculation
    params_update = np.linalg.solve(hessian, -grad)  # Solving Hx = -grad for x
    return params + params_update

# Initialization and Iteration
initial_params = np.array([-6.34058750e+01,  8.86102514e+04, -1.07374183e+09, -3.92595647e+06, -2.24466278e+08])
max_iterations = 100
convergence_threshold = 1e-6

params = initial_params
for iteration in range(max_iterations):
    params_new = newton_raphson_update(params, df_filtered)  # Use your data frame
    if np.linalg.norm(params_new - params) < convergence_threshold:
        print(f"Converged in {iteration + 1} iterations")
        break
    params = params_new

print(f"Final Parameters: {params}")


Final Parameters: [-2.68435335e+08  8.86102514e+04 -1.07374182e+09 -4.87202917e+08
  4.59333407e+08]


In [ ]:
### quadratic  theory !!! uncleaned

def utility(c, l, alpha , beta , gamma, delta,zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  alpha*c + beta *l - 0.5 * (gamma * c**2 + delta * l**2 + 2*zeta*c*l )

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha , beta , gamma, delta, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, alpha , beta , gamma, delta, zeta),
        utility(yds, 80 - lhw_scenario, alpha , beta , gamma, delta, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df_filtered ), epsilon)
    return grad

initial_params =  [ 0.5, 100, -5.36870872e+08, -4.47117970e+08,
  2.63168457e+08]
# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='BFGS')

print(f'Optimization result quadratic cleaned data : {result}')


In [71]:
### quadratic uncleaned

def utility(c, l, beta1, beta2, gamma1, gamma2, zeta):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * c**2 + gamma1 * l + gamma2 * l**2 + zeta * c * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2, zeta = params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2, zeta),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2, zeta)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 7.660e-01,  9.197e-01,  1.257e-01, -2.093e-03,  4.225e-05]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='BFGS', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :   message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 7907.682568073007
        x: [ 7.660e-01  9.197e-01  1.257e-01 -2.093e-03  4.225e-05]
      nit: 8
      jac: [ 0.000e+00  0.000e+00 -1.977e+00  7.083e+00  1.212e+02]
 hess_inv: [[ 1.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]
            [ 0.000e+00  1.000e+00 ...  0.000e+00  0.000e+00]
            ...
            [ 0.000e+00  0.000e+00 ...  3.015e-10 -1.498e-11]
            [ 0.000e+00  0.000e+00 ... -1.498e-11  1.125e-12]]
     nfev: 68
     njev: 56


In [82]:
df_filtered['consum'] = np.log(df_filtered['ils_udb_yds'])

df_filtered['logleis'] = np.log(80 - df_filtered['lhw'])

df_filtered[['idperson', 'ils_udb_yds', 'lhw' ,'logcon','loglhw','scenario' ,'consum', 'lhw_h0', 'lhw_h1','lhw_h2','lhw_h3','choice_made' ]]

c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\2897862135.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['consum'] = np.log(df_filtered['ils_udb_yds'])
C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\2897862135.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['logleis'] = np.log(80 - df_filtered['l

,idperson,ils_udb_yds,lhw,logcon,loglhw,scenario,consum,lhw_h0,lhw_h1,lhw_h2,lhw_h3,choice_made
0,87010001,825.40,42,3.737670,3.637586,h0,6.715868,0,7,42,54,0
1,87060001,840.40,44,3.784190,3.583519,h0,6.733878,0,11,44,50,0
2,87070001,1498.60,0,-inf,4.382027,h0,7.312287,0,7,27,52,1
3,87090001,819.57,37,3.610918,3.761200,h0,6.708780,0,10,37,51,0
4,87170002,790.40,40,3.688879,3.688879,h0,6.672539,0,7,40,51,0
...,...,...,...,...,...,...,...,...,...,...,...,...
18315,619730001,2233.56,46,3.828641,3.526361,h3,7.711352,0,7,46,63,0
18316,619850001,2207.17,0,-inf,4.382027,h3,7.699466,0,7,27,52,0
18317,619910001,2263.93,40,3.688879,3.688879,h3,7.724858,0,16,40,53,0
18318,620080001,2462.43,45,3.806662,3.555348,h3,7.808904,0,7,45,54,0


In [87]:
df_filtered[df_filtered['consum'] < 0]

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,lhw_scenario,du/dc,du/dL,du1/dc,du1/dL,consu,consum,logcon,loglhw,logleis
27,905700,90570003,90570002,90570001,0,905700,90570003,20,0,0,...,0,4.210695e+10,3.576914e+10,0.772970,1.079140,-inf,-inf,-inf,4.382027,4.382027
106,1215101,121510101,0,0,0,1215101,121510003,25,1,0,...,0,2.105348e+10,1.788442e+10,0.769485,0.999420,-inf,-inf,3.688879,3.688879,3.688879
576,1988300,198830002,0,0,198830001,1988300,198830002,74,0,0,...,0,4.210695e+10,3.576914e+10,0.772970,1.079140,-inf,-inf,-inf,4.382027,4.382027
596,2026600,202660002,0,202660001,0,2026600,202660002,23,1,0,...,0,2.105348e+10,1.788442e+10,0.769485,0.999420,-inf,-inf,3.688879,3.688879,3.688879
772,2240900,224090004,0,0,0,2240900,224090004,21,1,0,...,0,2.105348e+10,1.788442e+10,0.769485,0.999420,-inf,-inf,3.688879,3.688879,3.688879
961,2509700,250970003,250970002,250970001,0,2509700,250970003,23,1,0,...,0,4.210695e+10,3.576914e+10,0.772970,1.079140,-inf,-inf,-inf,4.382027,4.382027
988,2543400,254340002,0,0,254340001,2543400,254340002,66,0,0,...,0,2.210615e+10,1.877866e+10,0.769659,1.003406,-inf,-inf,3.637586,3.737670,3.737670
1111,2713800,271380001,0,0,0,2713800,271380001,23,1,0,...,0,2.105348e+10,1.788442e+10,0.769485,0.999420,-inf,-inf,3.688879,3.688879,3.688879
1345,3358900,335890002,335890001,0,0,3358900,335890002,25,0,0,...,0,2.105348e+10,1.788442e+10,0.769485,0.999420,-inf,-inf,3.688879,3.688879,3.688879
2160,4417100,441710001,441710003,441710002,0,4417100,441710001,22,1,0,...,0,2.150445e+10,1.832654e+10,0.674145,0.999493,-0.174353,-0.174353,3.688879,3.688879,3.688879


In [86]:
### translog uncleaned

def utility(c, l, beta1, beta2):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * l

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2 = params
    utilities = [
        utility(yds, np.log(160 - lhw_actual), beta1, beta2),
        utility(yds, np.log(160- lhw_scenario), beta1, beta2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['consum'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 7.660,  9.197]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered,), method='BFGS')

print(f'Optimization result quadratic cleaned data : {result}')


C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\820414389.py:24: RuntimeWarning: invalid value encountered in subtract
  log_probabilities = utilities - logsumexp(utilities)


Optimization result quadratic cleaned data :   message: NaN result encountered.
  success: False
   status: 3
      fun: nan
        x: [ 7.660e+00  9.197e+00]
      nit: 0
      jac: [       nan        nan]
 hess_inv: [[1 0]
            [0 1]]
     nfev: 3
     njev: 1


In [94]:
# Identify individuals with any negative 'ils_udb_yds' values
negative_c_ids = df_filtered[df_filtered['consum'] < 0]['idperson'].unique()
# Identify individuals with any negative 'ils_udb_yds' values
negative_l_ids = df_filtered[df_filtered['lhw'] > 80 ]['idperson'].unique()


print(f"Number of individuals with negative leisure: {len(negative_l_ids)}")
print(f"Number of individuals with negative consumption: {len(negative_c_ids)}")


Number of individuals with negative leisure: 0
Number of individuals with negative consumption: 30


In [95]:
# Filter long_df to exclude all rows belonging to individuals identified in step 1
df_filtered1 = df_filtered[~df_filtered['idperson'].isin(negative_c_ids)]


# Verify the removal
print(f"Original dataframe size: {df_filtered.shape}")
print(f"Filtered dataframe size: {df_filtered1.shape}")


Original dataframe size: (18308, 359)
Filtered dataframe size: (18188, 359)


In [96]:
### translog uncleaned

def utility(c, l, beta1, beta2):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * l

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2 = params
    utilities = [
        utility(yds, np.log(160 - lhw_actual), beta1, beta2),
        utility(yds, np.log(160- lhw_scenario), beta1, beta2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['consum'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 7.660,  9.197]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered1,), method='BFGS')

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :   message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 11196.25088148456
        x: [ 7.660e+00 -4.216e+00]
      nit: 10
      jac: [ 1.221e-04  1.514e-02]
 hess_inv: [[ 4.812e-05 -5.125e-07]
            [-5.125e-07  5.493e-09]]
     nfev: 186
     njev: 59


In [97]:
### translog uncleaned

def utility(c, l, beta1, beta2):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * l

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2 = params
    utilities = [
        utility(yds, np.log(160 - lhw_actual), beta1, beta2),
        utility(yds, np.log(160- lhw_scenario), beta1, beta2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['consum'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 1.660, .216]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered1,),  method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 11196.250888391498
       x: [ 5.660e+00 -4.216e+00]
     nit: 1
     jac: [ 0.000e+00 -5.505e-02]
    nfev: 2
    njev: 3
    nhev: 0


In [98]:
### translog uncleaned

def utility(c, l, beta1, beta2):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * l

# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2 = params
    utilities = [
        utility(yds, np.log(160 - lhw_actual), beta1, beta2),
        utility(yds, np.log(160- lhw_scenario), beta1, beta2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['consum'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Jacobian 

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p,df ), epsilon)
    return grad

initial_params =  [ 1.660, .216]

# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df_filtered1,),  method='Newton-CG', jac=jacobian)

print(f'Optimization result quadratic cleaned data : {result}')


Optimization result quadratic cleaned data :  message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 11196.251439362537
       x: [ 1.660e+00 -4.213e+00]
     nit: 46
     jac: [ 0.000e+00  3.684e-01]
    nfev: 461
    njev: 507
    nhev: 0


In [102]:
### quadratic  theory !!! rapjhson params


def utility(c, l, beta1, beta2):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return  beta1 * c + beta2 * l


# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2 = params
    utilities = [
        utility(yds, np.log( 80- lhw_actual), beta1, beta2),
        utility(yds, np.log( 80 - lhw_scenario), beta1, beta2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]


def calc_hessian(params, df):
    # This function should compute the Hessian matrix of your total likelihood function
    # with respect to the parameters.
    # For now, it returns a placeholder identity matrix of appropriate size
    return np.eye(len(params))

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['consum'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


# Newton-Raphson Update Function
def newton_raphson_update(params, df):
    grad = jacobian(params, df)  # Use your existing gradient calculation
    hessian = calc_hessian(params, df)  # Placeholder for your Hessian calculation
    params_update = np.linalg.solve(hessian, -grad)  # Solving Hx = -grad for x
    return params + params_update

# Initialization and Iteration
initial_params = np.array([ 1.660e+00, -4.213e+00])
max_iterations = 100
convergence_threshold = 1e-6

params = initial_params
for iteration in range(max_iterations):
    params_new = newton_raphson_update(params, df_filtered)  # Use your data frame
    if np.linalg.norm(params_new - params) < convergence_threshold:
        print(f"Converged in {iteration + 1} iterations")
        break
    params = params_new

print(f"Final Parameters: {params}")


C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\3222209728.py:26: RuntimeWarning: invalid value encountered in subtract
  log_probabilities = utilities - logsumexp(utilities)


Final Parameters: [nan nan]


In [103]:
## Trnslog 
def utility(params, row):
    # Unpacking parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]
    # Calculating the logs of income and leisure
    log_y = np.log(row['ils_udb_yds'])
    log_l = np.log(80 - row['lhw'])
    # Translog utility function
    return alpha * log_y + beta * log_l + 0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + gamma_yl * log_y * log_l

def ind_likelihood(params, group):
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else None
    if chosen_utility is not None:
        # Only consider scenarios where a choice is made
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  # Negative log likelihood
    else:
        return 0

def total_likelihood(params, df):
    # Grouping by individual and calculating likelihood per group
    total_ll = df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()
    return total_ll


initial_params = [0.1, 0.1, 0.01, 0.01, 0.01]  # Example: alpha, beta, gamma_yy, gamma_ll, gamma_yl

# Optimization
result = minimize(fun=total_likelihood, x0=initial_params, args=(df_filtered,), method='L-BFGS-B')
print(result)


C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\3090629599.py:6: RuntimeWarning: divide by zero encountered in log
  log_y = np.log(row['ils_udb_yds'])
C:\Users\hisham\AppData\Local\Temp\ipykernel_9820\3090629599.py:9: RuntimeWarning: invalid value encountered in scalar add
  return alpha * log_y + beta * log_l + 0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + gamma_yl * log_y * log_l


  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 4173.471234271588
        x: [ 6.007e+01  1.000e-01 -3.313e+00  1.001e-02 -9.230e+00]
      nit: 24
      jac: [-9.095e-05 -1.819e-04  1.910e-03 -9.095e-05  1.273e-03]
     nfev: 186
     njev: 31
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>


In [104]:
## Trnslog 
def utility(params, row):
    # Unpacking parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]
    # Calculating the logs of income and leisure
    log_y = np.log(row['ils_udb_yds'])
    log_l = np.log(80 - row['lhw'])
    # Translog utility function
    return alpha * log_y + beta * log_l + 0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + gamma_yl * log_y * log_l

def ind_likelihood(params, group):
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else None
    if chosen_utility is not None:
        # Only consider scenarios where a choice is made
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  # Negative log likelihood
    else:
        return 0

def total_likelihood(params, df):
    # Grouping by individual and calculating likelihood per group
    total_ll = df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()
    return total_ll


initial_params = [ 6.007e+01,  1.000e-01, -3.313e+00,  1.001e-02, -9.230e+00]  # Example: alpha, beta, gamma_yy, gamma_ll, gamma_yl

# Optimization
result = minimize(fun=total_likelihood, x0=initial_params, args=(df_filtered1,), method='L-BFGS-B')
print(result)


  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 4171.678418092324
        x: [ 6.007e+01  9.982e-02 -3.312e+00  9.875e-03 -9.232e+00]
      nit: 27
      jac: [ 1.091e-03 -1.819e-04  8.004e-03  0.000e+00  4.547e-03]
     nfev: 186
     njev: 31
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>


In [105]:
    ## Trnslog 
def utility(params, row):
        # Unpacking parameters
        alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]
        # Calculating the logs of income and leisure
        log_y = np.log(row['ils_udb_yds'])
        log_l = np.log(80 - row['lhw'])
        # Translog utility function
        return alpha * log_y + beta * log_l + 0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + gamma_yl * log_y * log_l

def ind_likelihood(params, group):
        utilities = group.apply(lambda row: utility(params, row), axis=1)
        chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else None
        if chosen_utility is not None:
            # Only consider scenarios where a choice is made
            log_prob_chosen = chosen_utility - logsumexp(utilities)
            return -log_prob_chosen  # Negative log likelihood
        else:
            return 0

def total_likelihood(params, df):
        # Grouping by individual and calculating likelihood per group
        total_ll = df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()
        return total_ll


initial_params = [ 6.007e+01,  1.000e-01, -3.313e+00,  1.001e-02, -9.230e+00]  # Example: alpha, beta, gamma_yy, gamma_ll, gamma_yl

    # Optimization
result = minimize(fun=total_likelihood, x0=initial_params, args=(df_filtered1,), method='Newton-CG', jac=jacobian)
print(result)


 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 4171.678527433941
       x: [ 6.007e+01  1.000e-01 -3.313e+00  1.001e-02 -9.230e+00]
     nit: 2
     jac: [ 3.998e-02  6.104e-05  1.673e-01  0.000e+00  2.021e-01]
    nfev: 4
    njev: 14
    nhev: 0


In [106]:
4171.678418092324 - 4171.678527433941

-0.00010934161673503695

In [121]:
def utility_extended(params, row):
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    log_y = np.log(row['ils_udb_yds'])
    log_l = np.log(80 - row['lhw'])
    utility_value = alpha * log_y + beta * log_l + \
                    0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + \
                    gamma_yl * log_y * log_l
    mu_y = alpha + gamma_yy * log_y + gamma_yl* log_l   # Marginal utility of income
    mu_l = beta + gamma_ll * log_l + gamma_yl * log_y  # Marginal utility of leisure
    return utility_value, mu_y, mu_l

df_filtered1[['utility', 'mu_y', 'mu_l']] = df_filtered1.apply(lambda row: utility_extended(result.x, row), axis=1, result_type="expand")


In [131]:
# Calculate probabilities for each person based on utility values
def calculate_probabilities_corrected(group):
    utilities = group['utility'].values
    exp_utilities = np.exp(utilities - np.max(utilities))
    probabilities = exp_utilities / np.sum(exp_utilities)
    return pd.Series(probabilities, index=group.index)

# Applying the function correctly
df_filtered1['probability'] = df_filtered1.groupby('idperson').apply(lambda group: calculate_probabilities_corrected(group)).reset_index(level=0, drop=True)


# Calculate the max probability for each group
max_probs = df_filtered1.groupby('idperson')['probability'].transform('max')

# Assign predicted choice based on whether the probability equals the max probability within its group
df_filtered1['predicted_choice'] = (df_filtered1['probability'] == max_probs).astype(int)



In [133]:
# Assuming 'scenario' column represents actual choice made and is in readable format (h0, h1, etc.)
actual_choice = df_filtered1[df_filtered1['choice_made'] == 1]['scenario']
predicted_choice = df_filtered1[df_filtered1['predicted_choice'] == 1]['scenario']

transition_matrix = pd.crosstab(actual_choice, predicted_choice, margins=True, margins_name='Total')
print(transition_matrix)


scenario   h0   h1   h2   h3  Total
scenario                           
h0        198    0    0    0    198
h1          0  190    0    0    190
h2          0    0  911    0    911
h3          0    0    0  454    454
Total     198  190  911  454   1753


In [139]:
print(df_filtered1['choice_made'].sum())


4547


In [138]:
print(df_filtered1.groupby('idperson')['scenario'].count().describe())
print(df_filtered1['scenario'].value_counts())


count    4547.0
mean        4.0
std         0.0
min         4.0
25%         4.0
50%         4.0
75%         4.0
max         4.0
Name: scenario, dtype: float64
scenario
h0    4547
h1    4547
h2    4547
h3    4547
Name: count, dtype: int64


In [148]:
# Assuming `df_filtered1` has columns: 'idperson', 'scenario', 'choice_made', 'predicted_choice'
# And 'scenario' column contains the scenario labels ('h0', 'h1', 'h2', 'h3')

# Marking actual choice scenarios (assuming 'choice_made' directly indicates this)
df_filtered1['actual_choice'] = df_filtered1.apply(lambda x: x['scenario'] if x['choice_made'] == 1 else None, axis=1)
df_filtered1['actual_choice'] = df_filtered1.groupby('idperson')['actual_choice'].ffill().bfill()

# Identifying the predicted choice scenario
df_filtered1['predicted_choice_scenario'] = df_filtered1.apply(lambda x: x['scenario'] if x['predicted_choice'] == 1 else None, axis=1)
df_filtered1['predicted_choice_scenario'] = df_filtered1.groupby('idperson')['predicted_choice_scenario'].ffill().bfill()

# Ensure there's no individual without a predicted scenario
assert df_filtered1['predicted_choice_scenario'].isnull().sum() == 0, "Some individuals don't have a predicted scenario."


In [155]:
no_predicted_choice_ids = df_filtered1.groupby('idperson')['choice_made'].max()
no_predicted_choice_ids = no_predicted_choice_ids[no_predicted_choice_ids == 0].index.tolist()

print("ID numbers of people with no predicted choice:", no_predicted_choice_ids)


ID numbers of people with no predicted choice: []


In [159]:
def calculate_marginal_utility(df, params):
    df['log_y'] = np.log(df['ils_udb_yds'])
    df['log_l'] = np.log(80 - df['lhw'])
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]

    df['MU_y'] = alpha + gamma_yy * df['log_y'] + gamma_yl * df['log_l']
    df['MU_l'] = beta + gamma_ll * df['log_l'] + gamma_yl * df['log_y']
    
    return df

# Assuming params contains your estimated parameters
params = [6.007e+01, 1.000e-01, -3.313e+00, 1.001e-02, -9.230e+00]
df_filtered1 = calculate_marginal_utility(df_filtered1, params)

# Now df_filtered1 contains 'MU_y' and 'MU_l' for every point


In [160]:
# Count of people with negative marginal utility of leisure at chosen points
negative_MU_l_chosen = df_filtered1[(df_filtered1['choice_made'] == 1) & (df_filtered1['MU_l'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of leisure at predicted points
negative_MU_l_predicted = df_filtered1[(df_filtered1['predicted_choice'] == 1) & (df_filtered1['MU_l'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of income at chosen points
negative_MU_y_chosen = df_filtered1[(df_filtered1['choice_made'] == 1) & (df_filtered1['MU_y'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of income at predicted points
negative_MU_y_predicted = df_filtered1[(df_filtered1['predicted_choice'] == 1) & (df_filtered1['MU_y'] < 0)]['idperson'].nunique()

print("Negative MU_l at chosen points:", negative_MU_l_chosen)
print("Negative MU_l at predicted points:", negative_MU_l_predicted)
print("Negative MU_y at chosen points:", negative_MU_y_chosen)
print("Negative MU_y at predicted points:", negative_MU_y_predicted)


Negative MU_l at chosen points: 4547
Negative MU_l at predicted points: 4547
Negative MU_y at chosen points: 1339
Negative MU_y at predicted points: 1450


In [167]:
# First, ensure 'predicted_choice' is correctly assigned in 'df_filtered1'
# This step assumes you have a method to assign 'predicted_choice' correctly based on your model's output

# For simplicity, let's rebuild 'predicted_choice_scenario' based on the highest utility or probability
# Assuming 'utility' or 'probability' column exists in 'df_filtered1' indicating model output

# Step 1: Assign predicted choice scenario based on the highest utility or probability
# This method assumes 'utility' column exists; adjust accordingly if using probabilities
df_filtered1['predicted_choice'] = df_filtered1.groupby('idperson')['utility'].transform(lambda x: x == x.max())

# Now, each individual should have exactly one 'predicted_choice' flagged
# Let's try assigning 'predicted_choice_scenario' again
df_filtered1['predicted_choice_scenario'] = df_filtered1.apply(
    lambda x: x['scenario'] if x['predicted_choice'] else None, axis=1
)

# Fill missing 'predicted_choice_scenario' within each group
df_filtered1['predicted_choice_scenario'] = df_filtered1.groupby('idperson')['predicted_choice_scenario'].ffill().bfill()

# Verify the assignment
missing_predictions_post_fix = df_filtered1[df_filtered1['choice_made'] == 1]['predicted_choice_scenario'].isnull().sum()
if missing_predictions_post_fix > 0:
    raise AssertionError(f"Post-fix, still missing predictions for {missing_predictions_post_fix} individuals.")

# Assuming no error is raised, proceed with creating the comparison table
# This time, ensure to use the adjusted dataset where predicted scenarios are guaranteed to be assigned
actual_choices_df = df_filtered1[df_filtered1['choice_made'] == 1].copy()
comparison_table = pd.crosstab(
    actual_choices_df['actual_choice'],
    actual_choices_df['predicted_choice_scenario'],
    margins=True,
    margins_name='Total'
)

print(comparison_table)


predicted_choice_scenario    h0   h1    h2   h3  Total
actual_choice                                         
h0                          205    0     0    0    205
h1                          237  258     0    0    495
h2                          595  651  2141    2   3389
h3                            1    1     2  454    458
Total                      1038  910  2143  456   4547


In [171]:
## Translog 
def utility(params, row):
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]
    # Calculating the logs 
    log_y = np.log(row['ils_udb_yds'])
    log_l = np.log(168 - row['lhw'])
    # Translog utility 
    return alpha * log_y + beta * log_l + 0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + gamma_yl * log_y * log_l

def ind_likelihood(params, group):
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else None
    if chosen_utility is not None:
        # we only consider scenarios where a choice is made
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  
    else:
        return 0

def total_likelihood(params, df):
    # Grouping by individual and calculating likelihood per group
    total_ll = df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()
    return total_ll


initial_params = [ 6.007e+01,  1.000e-01, -3.313e+00,  1.001e-02, -9.230e+00]  

result = minimize(fun=total_likelihood, x0=initial_params, args=(df_filtered1,), method='Newton-CG', jac=jacobian)
print(result)


 message: Warning: CG iterations didn't converge. The Hessian is not positive definite.
 success: False
  status: 3
     fun: 4545.192539657427
       x: [ 6.022e+01  1.000e-01 -2.418e+00  1.001e-02 -8.534e+00]
     nit: 5
     jac: [-1.810e+01 -1.221e-04 -1.788e+02  0.000e+00 -3.546e+01]
    nfev: 9
    njev: 213
    nhev: 0


In [177]:
heterogeneity_vars = df_filtered1[['dag',
 'dgn',
 'dcz',
 'deh',
 'dwt',
 'dms',
 'drgru',
 'drgur',
 'lcs',
 'lindi',
 'loc',
 'yivwg',
 'ils_b1_bsa',
 'il_bsa00', 'ils_b1_boa',
 'ils_b1_bsu',
 'ils_b1_bdi',
 'ils_b1_bun',
 'ils_b1_bhl',
 'ils_b1_bed',
 'ils_b1_bsa',
 'ils_b1_bcb',
 'ils_b1_bfa',
 'ils_b1_bho',
 'kfb',
 'kfbmy',
 'ils_udb_yiy',
 'ils_udb_ypp',
 'ils_udb_ypr',
 'ils_udb_ypt',
 'yptmp',
 'aca',
 'aco',
 'afc',
 'amrrm',
 'amrtn',
  'xmp',
 'xpp',
 'xhcmomi',
 'xhcmomc',
 'xhcmo',
 'xhcrt',
 'xcc',
 'xhc',
 'xed00',
 'xhl00',
 'xhcot'
]]

In [172]:
def utility(params, row, zeta_2, zeta_3):
    # Basic model parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params[:5]
    
    # Parameters for observed heterogeneity
    heterogeneity_params = params[5:-2]  # Assuming last two params are sigma_zeta2 and sigma_zeta3
    
    # Extracting observed heterogeneity variables
    heterogeneity_values = row[heterogeneity_vars]
    
    # Calculating observed heterogeneity effect
    observed_heterogeneity_effect = np.dot(heterogeneity_params, heterogeneity_values)
    
    # Logarithms of income and leisure
    log_y = np.log(row['ils_udb_yds'])
    log_l = np.log(168 - row['lhw'])
    
    # Translog utility function with observed and unobserved heterogeneity
    utility_val = (alpha + observed_heterogeneity_effect) * log_y + beta * log_l + \
                   0.5 * gamma_yy * log_y**2 + 0.5 * gamma_ll * log_l**2 + \
                   gamma_yl * log_y * log_l + \
                   zeta_2 * log_l + zeta_3 * log_l**2  # Example adjustment for unobserved heterogeneity
    
    return utility_val


In [176]:
def simulate_zetas(n, sigma_zeta2, sigma_zeta3):
    return np.random.normal(0, sigma_zeta2, n), np.random.normal(0, sigma_zeta3, n)

def ind_likelihood(params, row, n_simulations=100):
    sigma_zeta2, sigma_zeta3 = params[-2], params[-1]
    simulated_likelihoods = []
    
    for _ in range(n_simulations):
        zeta_2, zeta_3 = simulate_zetas(1, sigma_zeta2, sigma_zeta3)
        u = utility(params, row, zeta_2, zeta_3)
        simulated_likelihoods.append(u)
        
    avg_likelihood = np.mean(simulated_likelihoods)
    return -avg_likelihood  # Negative for minimization

def total_likelihood(params, df, n_simulations=100):
    total_ll = sum(df.apply(ind_likelihood, axis=1, args=(params, n_simulations)))
    return total_ll





In [178]:
# Number of observed heterogeneity variables
k = len(heterogeneity_vars)

# Initial parameter guesses for the base model parameters
base_params = [60.07, 0.1, -3.313, 0.01001, -9.23]

# Initial guesses for observed heterogeneity effects (set to zeros)
observed_heterogeneity_initials = [0] * k

# Initial standard deviations for unobserved heterogeneity components
sigma_zeta2_initial = 0.1
sigma_zeta3_initial = 0.1

# Combining all initial guesses into one list
initial_params = base_params + observed_heterogeneity_initials + [sigma_zeta2_initial, sigma_zeta3_initial]


In [179]:
# Assuming `initial_params` includes initial guesses for all parameters
# And `df` is your DataFrame with all necessary columns including `heterogeneity_vars`
result = minimize(lambda params: total_likelihood(params, df_filtered1), initial_params, method='L-BFGS-B')

print(result)

ValueError: scale < 0